# Camada Gold — `ecommerce_produtos` (Data Quality BI)

Este notebook lê:
- `squad1/dq_monitoring_logs` — logs de DQ gravados na raiz do container
- `squad1/silver/ecommerce_produtos` — Silver de produtos

e produz **6 tabelas Gold** em Delta físico em `squad1/gold/ecommerce_produtos/`
e espelha cada uma no **Azure SQL Server** (schema `squad1`, para consumo via Looker):

| # | Tabela Gold | Descrição |
|---|---|---|
| 1 | `gold_ecommerce_produtos_dq_resumo_por_regra` | % de falha por regra por dia |
| 2 | `gold_ecommerce_produtos_dq_resumo_por_tabela` | % de registros limpos por tabela por hora |
| 3 | `gold_ecommerce_produtos_dq_tendencia_diaria` | Score de qualidade diário (0–100) + variação Δ + tendência |
| 4 | `gold_ecommerce_produtos_dq_regras_criticas` | Regras com % de falha histórica acima do threshold (5%) |
| 5 | `gold_ecommerce_produtos_snapshot_validos` | Visão analítica dos produtos Silver válidos (precificação, marca, categoria) |
| 6 | `gold_ecommerce_produtos_dq_perfil_por_categoria` | Saúde de DQ segmentada por categoria de produto |

> **Ambiente**: Databricks Free Edition (Serverless).
> Toda I/O de Delta usa o mesmo mecanismo da Silver: `DataLakeServiceClient` para
> leitura de Parquet físico + escrita manual do `_delta_log` — sem `write_deltalake`,
> sem Rust, sem dependência de autenticação OAuth no sandbox.
> O espelhamento no SQL Server usa o conector nativo `sqlserver` do Spark.


## 1. Instalação de dependências

In [0]:
%pip install -q "deltalake==0.15.3" "pyarrow==14.0.1" python-dotenv azure-identity azure-storage-file-datalake


## 2. Imports, parâmetros e credenciais

In [0]:
import os
import uuid
import uuid as uuid_lib
import json
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
from io import BytesIO
from functools import reduce
from datetime import datetime, timezone
from dotenv import load_dotenv

from azure.identity import ClientSecretCredential
from azure.storage.filedatalake import DataLakeServiceClient

from pyspark.sql import functions as F
from pyspark.sql.window import Window

# ─── Contêiner e caminhos físicos ────────────────────────────────────────────
CONTAINER_SQUAD1 = "squad1"

# Fontes de leitura
CAMADA_DQ    = ""                  # raiz do container (sem prefixo de pasta)
ENTIDADE_DQ  = "dq_monitoring_logs"

CAMADA_SILVER  = "silver"
ENTIDADE_SILVER = "ecommerce_produtos"

# Destino Gold
CAMADA_GOLD   = "gold"
ENTIDADE_GOLD = "ecommerce_produtos"   # subpasta dentro de gold/

# ─── Nomes das 6 tabelas Gold ─────────────────────────────────────────────────
TABELA_GOLD_REGRA      = "gold_ecommerce_produtos_dq_resumo_por_regra"
TABELA_GOLD_TABELA     = "gold_ecommerce_produtos_dq_resumo_por_tabela"
TABELA_GOLD_TENDENCIA  = "gold_ecommerce_produtos_dq_tendencia_diaria"
TABELA_GOLD_CRITICAS   = "gold_ecommerce_produtos_dq_regras_criticas"
TABELA_GOLD_SNAPSHOT   = "gold_ecommerce_produtos_snapshot_validos"
TABELA_GOLD_CATEGORIA  = "gold_ecommerce_produtos_dq_perfil_por_categoria"

# Tabela-alvo referenciada nos logs de DQ (filtro)
TABELA_ALVO = "silver_ecommerce_produtos"

# Threshold para sinalizar regras críticas
THRESHOLD_FALHA_CRITICA = 5.0   # %

# Limite de preço para outlier (deve coincidir com R7 da Silver)
PRECO_LIMITE_OUTLIER = 5000.0

RUN_ID = str(uuid.uuid4())
print("RUN_ID:", RUN_ID)

# ─── Credenciais ADLS (Service Principal) ────────────────────────────────────
load_dotenv("/Workspace/Users/soaress.elias@gmail.com/merca-data-platform-categorias/.env")

CLIENT_ID            = os.getenv("ADLS_CLIENT_ID")
TENANT_ID            = os.getenv("ADLS_TENANT_ID")
CLIENT_SECRET        = os.getenv("ADLS_CLIENT_SECRET")
STORAGE_ACCOUNT_NAME = os.getenv("ADLS_STORAGE_ACCOUNT_NAME")

STORAGE_OPTIONS = {
    "account_name":  STORAGE_ACCOUNT_NAME,
    "client_id":     CLIENT_ID,
    "client_secret": CLIENT_SECRET,
    "tenant_id":     TENANT_ID,
}

credential = ClientSecretCredential(
    tenant_id=TENANT_ID,
    client_id=CLIENT_ID,
    client_secret=CLIENT_SECRET
)
service_client = DataLakeServiceClient(
    account_url=f"https://{STORAGE_ACCOUNT_NAME}.dfs.core.windows.net",
    credential=credential
)
file_system_squad1 = service_client.get_file_system_client(file_system=CONTAINER_SQUAD1)

# ─── Credenciais SQL Server ───────────────────────────────────────────────────
JDBC_HOSTNAME = os.getenv("SQL_HOST")
JDBC_DATABASE = os.getenv("SQL_DATABASE")
JDBC_USERNAME = os.getenv("SQL_USERNAME")
JDBC_PASSWORD = os.getenv("SQL_PASSWORD")

print(f"ADLS account   : {STORAGE_ACCOUNT_NAME}")
print(f"SQL Server host: {JDBC_HOSTNAME}")
print(f"SQL Database   : {JDBC_DATABASE}")
print(f"Origem DQ logs : {CONTAINER_SQUAD1}/{ENTIDADE_DQ}")
print(f"Origem Silver  : {CONTAINER_SQUAD1}/{CAMADA_SILVER}/{ENTIDADE_SILVER}")
print(f"Destino Gold   : {CONTAINER_SQUAD1}/{CAMADA_GOLD}/{ENTIDADE_GOLD}/<tabela>")
print("✅ Credenciais carregadas com sucesso.")


## 3. Funções auxiliares

Mesmo mecanismo de I/O da camada Silver de `ecommerce_produtos`:
leitura via `DataLakeServiceClient` (lê Parquet físico diretamente),
gravação com escrita manual de Parquet + `_delta_log` — sem `write_deltalake`,
sem Rust, compatível com o sandbox do Databricks Serverless.


In [0]:
# ─────────────────────────────────────────────────────────────────────────────
# Helpers de caminho
# ─────────────────────────────────────────────────────────────────────────────

def _pasta_fisica(camada: str, tabela: str) -> str:
    """Caminho relativo dentro do container squad1."""
    return f"{camada}/{tabela}" if camada else tabela


def _pasta_fisica_gold(nome_subtabela: str) -> str:
    """Caminho relativo para uma sub-tabela Gold em gold/ecommerce_produtos/<nome>."""
    return f"{CAMADA_GOLD}/{ENTIDADE_GOLD}/{nome_subtabela}"


# ─────────────────────────────────────────────────────────────────────────────
# Verificação de existência (via _delta_log no ADLS)
# ─────────────────────────────────────────────────────────────────────────────

def delta_existe_path(pasta_raiz: str) -> bool:
    """Verifica se existe _delta_log no caminho relativo informado."""
    try:
        pasta_log = f"{pasta_raiz}/_delta_log"
        paths = list(file_system_squad1.get_paths(path=pasta_log, recursive=False))
        return len(paths) > 0
    except Exception:
        return False


def delta_existe(camada: str, tabela: str, storage_opts: dict) -> bool:
    return delta_existe_path(_pasta_fisica(camada, tabela))


# ─────────────────────────────────────────────────────────────────────────────
# Leitura de Delta físico (lê todos os Parquet da pasta)
# ─────────────────────────────────────────────────────────────────────────────

def _ler_parquets_da_pasta(pasta_raiz: str):
    """Baixa todos os arquivos .parquet da pasta e retorna um DataFrame PySpark."""
    todos_paths = list(file_system_squad1.get_paths(path=pasta_raiz, recursive=True))
    arquivos_parquet = [
        p.name for p in todos_paths
        if not p.is_directory and p.name.endswith(".parquet")
    ]
    if not arquivos_parquet:
        raise Exception(f"Nenhum arquivo Parquet encontrado em {pasta_raiz}")

    print(f"  Lendo {len(arquivos_parquet)} arquivo(s) Parquet de '{pasta_raiz}'...")
    dfs = []
    for arq in arquivos_parquet:
        fc  = file_system_squad1.get_file_client(arq)
        raw = fc.download_file().readall()
        pdf = pd.read_parquet(BytesIO(raw))
        dfs.append(pdf)

    pdf_total = pd.concat(dfs, ignore_index=True)

    # Normaliza timezone em colunas datetime
    for col in pdf_total.columns:
        if pd.api.types.is_datetime64_any_dtype(pdf_total[col]):
            try:
                pdf_total[col] = pdf_total[col].dt.tz_localize(None)
            except TypeError:
                pdf_total[col] = pdf_total[col].dt.tz_convert(None)

    return spark.createDataFrame(pdf_total)


def ler_delta(camada: str, tabela: str, storage_opts: dict):
    pasta = _pasta_fisica(camada, tabela)
    if not delta_existe_path(pasta):
        raise Exception(f"Tabela Delta não encontrada: {pasta}")
    return _ler_parquets_da_pasta(pasta)


# ─────────────────────────────────────────────────────────────────────────────
# Upload de bytes para o ADLS
# ─────────────────────────────────────────────────────────────────────────────

def _upload_bytes(caminho_relativo: str, conteudo: bytes):
    fc = file_system_squad1.get_file_client(caminho_relativo)
    fc.create_file()
    fc.append_data(conteudo, offset=0, length=len(conteudo))
    fc.flush_data(len(conteudo))


# ─────────────────────────────────────────────────────────────────────────────
# Próxima versão do _delta_log
# ─────────────────────────────────────────────────────────────────────────────

def _proximo_version_delta(pasta_raiz: str) -> int:
    pasta_log = f"{pasta_raiz}/_delta_log"
    try:
        paths  = list(file_system_squad1.get_paths(path=pasta_log, recursive=False))
        jsons  = [p.name for p in paths if p.name.endswith(".json")]
        if not jsons:
            return 0
        versoes = [int(os.path.basename(p).replace(".json", "")) for p in jsons]
        return max(versoes) + 1
    except Exception:
        return 0


# ─────────────────────────────────────────────────────────────────────────────
# Gravação de Delta físico (sem write_deltalake / sem Rust)
# Tabelas Gold: sem particionamento (são agregações pequenas, prontas para BI).
# Modo padrão: overwrite (substituição completa a cada execução).
# ─────────────────────────────────────────────────────────────────────────────

def _arrow_type_to_delta(t) -> str:
    if pa.types.is_string(t) or pa.types.is_large_string(t): return "string"
    if pa.types.is_int32(t):    return "integer"
    if pa.types.is_int64(t):    return "long"
    if pa.types.is_float32(t):  return "float"
    if pa.types.is_float64(t):  return "double"
    if pa.types.is_boolean(t):  return "boolean"
    if pa.types.is_timestamp(t):return "timestamp"
    if pa.types.is_date32(t):   return "date"
    return "string"


def gravar_delta_gold(df, nome_subtabela: str) -> bool:
    """
    Grava um DataFrame Spark como Delta físico em
    gold/ecommerce_produtos/<nome_subtabela> via DataLakeServiceClient.
    Sem particionamento; modo overwrite (substituição total a cada run).
    """
    pasta_raiz = _pasta_fisica_gold(nome_subtabela)
    versao     = _proximo_version_delta(pasta_raiz)

    try:
        pdf = df.toPandas()

        # Normaliza datetimes
        for col in pdf.columns:
            if pd.api.types.is_datetime64_any_dtype(pdf[col]):
                try:
                    pdf[col] = pdf[col].dt.tz_localize(None)
                except TypeError:
                    pdf[col] = pdf[col].dt.tz_convert(None)

        # Grava Parquet
        nome_arquivo   = f"part-{str(uuid_lib.uuid4())[:8]}.snappy.parquet"
        caminho_parquet = f"{pasta_raiz}/{nome_arquivo}"
        buffer         = BytesIO()
        arrow_table    = pa.Table.from_pandas(pdf, preserve_index=False)
        pq.write_table(arrow_table, buffer, compression="snappy")
        _upload_bytes(caminho_parquet, buffer.getvalue())

        # Monta _delta_log
        schema_fields = [
            {"name": f.name, "type": _arrow_type_to_delta(f.type),
             "nullable": True, "metadata": {}}
            for f in arrow_table.schema
        ]
        delta_schema = {"type": "struct", "fields": schema_fields}
        ts_ms = int(datetime.now(timezone.utc).timestamp() * 1000)

        if versao == 0:
            linhas_log = [
                {"commitInfo": {
                    "timestamp": ts_ms,
                    "operation": "WRITE",
                    "operationParameters": {"mode": "Overwrite", "partitionBy": "[]"},
                    "isBlindAppend": False
                }},
                {"metaData": {
                    "id": str(uuid_lib.uuid4()),
                    "format": {"provider": "parquet", "options": {}},
                    "schemaString": json.dumps(delta_schema),
                    "partitionColumns": [],
                    "configuration": {},
                    "createdTime": ts_ms
                }},
                {"protocol": {"minReaderVersion": 1, "minWriterVersion": 2}},
            ]
        else:
            linhas_log = [
                {"commitInfo": {
                    "timestamp": ts_ms,
                    "operation": "WRITE",
                    "operationParameters": {"mode": "Overwrite", "partitionBy": "[]"},
                    "isBlindAppend": False
                }}
            ]

        # Entrada "add" para o arquivo gravado
        linhas_log.append({
            "add": {
                "path": nome_arquivo,
                "partitionValues": {},
                "size": len(buffer.getvalue()),
                "modificationTime": ts_ms,
                "dataChange": True
            }
        })

        nome_log     = f"{str(versao).zfill(20)}.json"
        caminho_log  = f"{pasta_raiz}/_delta_log/{nome_log}"
        conteudo_log = "\n".join(json.dumps(l) for l in linhas_log).encode("utf-8")
        _upload_bytes(caminho_log, conteudo_log)

        print(f"  [Delta OK] {pasta_raiz} | {len(pdf)} linhas | versão {versao}")
        return True

    except Exception as e:
        import traceback
        print(f"  [Delta ERRO] {pasta_raiz}: {e}")
        traceback.print_exc()
        return False


# ─────────────────────────────────────────────────────────────────────────────
# Espelhamento no Azure SQL Server
# ─────────────────────────────────────────────────────────────────────────────

def salvar_sql_server(df, nome_tabela: str):
    """
    Espelha um DataFrame no Azure SQL Server via conector nativo Spark 'sqlserver'.
    Schema de destino: squad1. Modo: overwrite (truncate + insert).
    Compatível com Databricks Serverless (sem JAR externo).
    """
    try:
        df.write \
            .format("sqlserver") \
            .option("host",     JDBC_HOSTNAME) \
            .option("port",     "1433") \
            .option("database", JDBC_DATABASE) \
            .option("dbtable",  f"squad1.{nome_tabela}") \
            .option("user",     JDBC_USERNAME) \
            .option("password", JDBC_PASSWORD) \
            .option("trustServerCertificate", "true") \
            .mode("overwrite") \
            .save()
        print(f"  [SQL OK] squad1.{nome_tabela}")
    except Exception as e:
        print(f"  [SQL ERRO] squad1.{nome_tabela}: {e}")


# ─────────────────────────────────────────────────────────────────────────────
# Wrapper unificado: ADLS (Delta) + SQL Server
# ─────────────────────────────────────────────────────────────────────────────

def publicar_gold(df, nome_subtabela: str):
    """Grava no ADLS (Delta físico) e no SQL Server em sequência."""
    print(f"\n>>> Publicando: {nome_subtabela}")
    gravar_delta_gold(df, nome_subtabela)
    salvar_sql_server(df, nome_subtabela)
    return df


## 4. Leitura das fontes (DQ Logs + Silver de produtos)

In [0]:
# ─── 4a. DQ Monitoring Logs ───────────────────────────────────────────────────
print("Lendo dq_monitoring_logs...")
df_dq_raw = ler_delta(camada=CAMADA_DQ, tabela=ENTIDADE_DQ, storage_opts=STORAGE_OPTIONS)

# Exibe um resumo das tabelas presentes nos logs (apenas para diagnóstico)
display(df_dq_raw.groupBy("tabela").count())

# Filtra apenas os logs da tabela de interesse
df_dq = df_dq_raw.where(F.col("tabela") == TABELA_ALVO)

total_dq = df_dq.count()
print(f"  Registros de DQ para '{TABELA_ALVO}': {total_dq}")
if total_dq == 0:
    raise Exception(
        f"Nenhum log de DQ encontrado para tabela='{TABELA_ALVO}'. "
        "Execute o notebook Silver de ecommerce_produtos antes deste notebook Gold."
    )

# ─── 4b. Silver ecommerce_produtos ───────────────────────────────────────────
print("\nLendo Silver ecommerce_produtos...")
df_silver = ler_delta(camada=CAMADA_SILVER, tabela=ENTIDADE_SILVER, storage_opts=STORAGE_OPTIONS)
total_silver = df_silver.count()
print(f"  Registros na Silver: {total_silver}")
if total_silver == 0:
    raise Exception(
        f"A tabela Silver '{CAMADA_SILVER}/{ENTIDADE_SILVER}' está vazia. "
        "Certifique-se de que o notebook Silver foi executado e gerou dados."
    )

# Esquema rápido para referência
print("\nColunas da Silver:")
for c in df_silver.columns:
    print(f"  {c}")

display(df_dq.limit(5))
display(df_silver.limit(5))


## 5. Gold 1 — `gold_ecommerce_produtos_dq_resumo_por_regra`

**Objetivo BI**: monitorar a % de falha de cada regra de DQ por dia.
Permite identificar regressões regra a regra ao longo do tempo e comparar
a saúde de regras críticas vs. regras de aviso.

Colunas:
- `data_execucao` — data (YYYY-MM-DD) extraída de `timestamp_execucao`
- `regra`, `severidade`
- `total_execucoes` — quantidade de entradas de log para essa regra no dia
- `total_falhos`, `total_registros`
- `pct_falha` — % de registros que falharam nessa regra no dia
- `status_dia` — FAIL se houve ao menos 1 FAIL no dia, PASS caso contrário
- `gold_generated_at`


In [0]:
df_gold_regra = (
    df_dq
    .withColumn("data_execucao", F.to_date(F.col("timestamp_execucao")))
    .groupBy("data_execucao", "regra", "severidade")
    .agg(
        F.count("*").cast("long").alias("total_execucoes"),
        F.sum("qtd_registros_falhos").cast("long").alias("total_falhos"),
        F.sum("qtd_registros_total").cast("long").alias("total_registros"),
        F.max(F.when(F.col("status") == "FAIL", 1).otherwise(0)).alias("_teve_fail"),
    )
    .withColumn(
        "pct_falha",
        F.round(
            F.when(F.col("total_registros") > 0,
                   (F.col("total_falhos") / F.col("total_registros")) * 100
            ).otherwise(F.lit(0.0)),
            4
        )
    )
    .withColumn("status_dia",
        F.when(F.col("_teve_fail") == 1, "FAIL").otherwise("PASS"))
    .drop("_teve_fail")
    .withColumn("gold_generated_at", F.current_timestamp())
    .orderBy("data_execucao", "regra")
)

print(f"Linhas geradas: {df_gold_regra.count()}")
display(df_gold_regra)


## 6. Gold 2 — `gold_ecommerce_produtos_dq_resumo_por_tabela`

**Objetivo BI**: monitorar a % de registros **limpos** (sem falha em nenhuma regra)
por tabela por hora — ideal para dashboards de saúde em tempo quase-real e alertas de SLA.

Um arquivo (batch) é considerado "limpo" quando **todas** as regras retornaram PASS para ele.

Colunas:
- `tabela`, `data_hora` — janela horária (YYYY-MM-DD HH:00:00)
- `total_registros_avaliados`, `total_registros_limpos`
- `pct_limpos` — % de registros limpos sobre o total avaliado
- `arquivos_com_falha` — quantos arquivos/batches tiveram ao menos 1 regra FAIL nessa hora
- `gold_generated_at`


In [0]:
# Agrega por (tabela, hora, arquivo): marca como "limpo" se todas as regras passaram
df_arq_hora = (
    df_dq
    .withColumn("data_hora", F.date_trunc("hour", F.col("timestamp_execucao")))
    .groupBy("tabela", "data_hora", "arquivo_origem")
    .agg(
        F.max("qtd_registros_total").alias("qtd_registros"),
        F.sum(F.when(F.col("status") == "FAIL", 1).otherwise(0)).alias("regras_que_falharam"),
    )
    .withColumn("arquivo_limpo",
        F.when(F.col("regras_que_falharam") == 0, 1).otherwise(0))
)

df_gold_tabela = (
    df_arq_hora
    .groupBy("tabela", "data_hora")
    .agg(
        F.sum("qtd_registros").cast("long").alias("total_registros_avaliados"),
        F.sum(
            F.when(F.col("arquivo_limpo") == 1, F.col("qtd_registros")).otherwise(0)
        ).cast("long").alias("total_registros_limpos"),
        F.sum(F.when(F.col("arquivo_limpo") == 0, 1).otherwise(0))
         .cast("long").alias("arquivos_com_falha"),
    )
    .withColumn(
        "pct_limpos",
        F.round(
            F.when(F.col("total_registros_avaliados") > 0,
                   (F.col("total_registros_limpos") / F.col("total_registros_avaliados")) * 100
            ).otherwise(F.lit(0.0)),
            4
        )
    )
    .withColumn("gold_generated_at", F.current_timestamp())
    .orderBy("tabela", "data_hora")
)

print(f"Linhas geradas: {df_gold_tabela.count()}")
display(df_gold_tabela)


## 7. Gold 3 — `gold_ecommerce_produtos_dq_tendencia_diaria`

**Objetivo BI**: acompanhar o **score de qualidade** (0–100) dia a dia e sua
variação (Δ) em relação ao dia anterior — ideal para gráficos de tendência,
alertas de regressão e análise de impacto de mudanças no pipeline.

Colunas:
- `data_execucao`
- `score_qualidade` — 100 − média da % de falha entre todas as regras do dia
- `pct_falha_media_dia` — média ponderada de % de falha por regra
- `total_registros_dia`, `total_falhos_dia`
- `qtd_regras_avaliadas`, `qtd_regras_com_falha`
- `delta_score` — variação vs. dia anterior (NULL no 1º dia)
- `tendencia` — MELHORA / PIORA / ESTAVEL / PRIMEIRO_DIA
- `gold_generated_at`


In [0]:
df_diario = (
    df_dq
    .withColumn("data_execucao", F.to_date(F.col("timestamp_execucao")))
    .groupBy("data_execucao")
    .agg(
        F.sum("qtd_registros_total").cast("long").alias("total_registros_dia"),
        F.sum("qtd_registros_falhos").cast("long").alias("total_falhos_dia"),
        F.countDistinct("regra").alias("qtd_regras_avaliadas"),
        F.sum(F.when(F.col("status") == "FAIL", 1).otherwise(0))
         .cast("long").alias("qtd_regras_com_falha"),
        F.avg(
            F.when(F.col("qtd_registros_total") > 0,
                   (F.col("qtd_registros_falhos") / F.col("qtd_registros_total")) * 100
            ).otherwise(F.lit(0.0))
        ).alias("pct_falha_media_dia"),
    )
    .withColumn(
        "score_qualidade",
        F.round(F.greatest(F.lit(0.0), F.lit(100.0) - F.col("pct_falha_media_dia")), 4)
    )
    .withColumn("pct_falha_media_dia", F.round(F.col("pct_falha_media_dia"), 4))
)

w_dia = Window.orderBy("data_execucao")

df_gold_tendencia = (
    df_diario
    .withColumn("score_anterior", F.lag("score_qualidade", 1).over(w_dia))
    .withColumn("delta_score",
        F.round(F.col("score_qualidade") - F.col("score_anterior"), 4))
    .withColumn(
        "tendencia",
        F.when(F.col("score_anterior").isNull(), "PRIMEIRO_DIA")
         .when(F.col("delta_score") >  0.5,      "MELHORA")
         .when(F.col("delta_score") < -0.5,      "PIORA")
         .otherwise("ESTAVEL")
    )
    .drop("score_anterior")
    .withColumn("gold_generated_at", F.current_timestamp())
    .orderBy("data_execucao")
)

print(f"Linhas geradas: {df_gold_tendencia.count()}")
display(df_gold_tendencia)


## 8. Gold 4 — `gold_ecommerce_produtos_dq_regras_criticas`

**Objetivo BI**: ranking consolidado das regras com % de falha histórica acima do
threshold de **5%** (`THRESHOLD_FALHA_CRITICA`). Prioriza triagem pela equipe de DQ —
regras críticas (severidade Critica) com alta taxa de falha recebem `status_alerta = CRITICO`.

Colunas:
- `regra`, `severidade`
- `total_execucoes_historico`, `total_falhos_historico`, `total_registros_historico`
- `pct_falha_historico`
- `dias_com_falha` — quantidade de dias distintos com ao menos 1 FAIL
- `primeiro_registro_falha`, `ultimo_registro_falha`
- `status_alerta` — CRITICO / ATENCAO / OK
- `threshold_pct`, `gold_generated_at`


In [0]:
df_gold_criticas = (
    df_dq
    .withColumn("data_execucao", F.to_date(F.col("timestamp_execucao")))
    .groupBy("regra", "severidade")
    .agg(
        F.count("*").cast("long").alias("total_execucoes_historico"),
        F.sum("qtd_registros_falhos").cast("long").alias("total_falhos_historico"),
        F.sum("qtd_registros_total").cast("long").alias("total_registros_historico"),
        F.countDistinct(
            F.when(F.col("status") == "FAIL", F.col("data_execucao"))
        ).alias("dias_com_falha"),
        F.min("timestamp_execucao").alias("primeiro_registro_falha"),
        F.max("timestamp_execucao").alias("ultimo_registro_falha"),
    )
    .withColumn(
        "pct_falha_historico",
        F.round(
            F.when(F.col("total_registros_historico") > 0,
                   (F.col("total_falhos_historico") / F.col("total_registros_historico")) * 100
            ).otherwise(F.lit(0.0)),
            4
        )
    )
    .withColumn(
        "status_alerta",
        F.when(
            (F.col("pct_falha_historico") >= THRESHOLD_FALHA_CRITICA) &
            (F.col("severidade") == "Critica"),
            "CRITICO"
        )
        .when(F.col("pct_falha_historico") >= THRESHOLD_FALHA_CRITICA, "ATENCAO")
        .otherwise("OK")
    )
    .withColumn("threshold_pct", F.lit(THRESHOLD_FALHA_CRITICA))
    .withColumn("gold_generated_at", F.current_timestamp())
    .orderBy(F.col("pct_falha_historico").desc())
)

print(f"Linhas geradas: {df_gold_criticas.count()}")
display(df_gold_criticas)


## 9. Gold 5 — `gold_ecommerce_produtos_snapshot_validos`

**Objetivo BI**: visão analítica dos produtos Silver **válidos** (`silver_linha_valida = true`),
enriquecida com métricas de precificação, marca e indicadores calculados.
Base para análises de cobertura de catálogo, pricing e sell-out no Looker.

Colunas:
- Campos de negócio: `sku`, `nome_produto`, `nome_marca`, `id_categoria`,
  `preco_lista`, `unidade_medida`, `is_ativo`, `tem_venda_recente`
- `faixa_preco` — Baixo / Medio / Alto / Premium (baseado em quartis fixos)
- `flag_preco_outlier` — TRUE se `preco_lista > PRECO_LIMITE_OUTLIER`
- `qtd_regras_falhas` — total de regras que falharam por registro (sempre 0 aqui, pois só válidos)
- `score_registro` — 100 × (1 − qtd_regras_falhas / 10)
- `silver_processed_at`, `gold_generated_at`


In [0]:
# Identifica colunas de flags de regra presentes na Silver
flags_regra = [c for c in df_silver.columns if c.startswith("r") and c.endswith("_falhou")]
n_regras    = max(len(flags_regra), 1)

# Expressão para contar regras que falharam por linha
if flags_regra:
    expr_qtd_falhas = reduce(
        lambda a, b: a + b,
        [F.when(F.col(c), 1).otherwise(0).cast("int") for c in flags_regra]
    )
else:
    expr_qtd_falhas = F.lit(0).cast("int")

# Colunas de negócio desejadas (usa .get para ser robusto a variações de schema)
colunas_negocio = [
    c for c in [
        "sku", "nome_produto", "nome_marca", "id_categoria",
        "preco_lista", "unidade_medida", "is_ativo", "tem_venda_recente",
        "silver_processed_at"
    ] if c in df_silver.columns
]

df_gold_snapshot = (
    df_silver
    .where(F.col("silver_linha_valida") == True)
    .select(*colunas_negocio, *flags_regra)
    # Faixa de preço
    .withColumn(
        "faixa_preco",
        F.when(F.col("preco_lista") <= 50,   "Baixo")
         .when(F.col("preco_lista") <= 200,  "Medio")
         .when(F.col("preco_lista") <= 1000, "Alto")
         .otherwise("Premium")
    )
    # Flag de outlier de preço
    .withColumn(
        "flag_preco_outlier",
        F.when(F.col("preco_lista") > PRECO_LIMITE_OUTLIER, True).otherwise(False)
    )
    # Score por registro
    .withColumn("qtd_regras_falhas", expr_qtd_falhas)
    .withColumn(
        "score_registro",
        F.round(F.lit(100.0) - (F.col("qtd_regras_falhas") / F.lit(n_regras)) * 100.0, 2)
    )
    # Remove flags individuais (detalhes internos da Silver)
    .drop(*flags_regra)
    .withColumn("gold_generated_at", F.current_timestamp())
    .orderBy("id_categoria", "nome_marca", "sku")
)

print(f"Produtos válidos no snapshot: {df_gold_snapshot.count()}")
display(df_gold_snapshot.limit(20))


## 10. Gold 6 — `gold_ecommerce_produtos_dq_perfil_por_categoria`

**Objetivo BI**: saúde de DQ segmentada por `id_categoria`.
Permite identificar quais categorias concentram mais produtos inválidos,
quais regras são mais problemáticas por categoria e priorizar ações corretivas.

Esta tabela é derivada diretamente da Silver (não dos logs de DQ), portanto
reflete o estado atual dos dados — não o histórico de execuções.

Colunas:
- `id_categoria`
- `total_produtos` — quantidade total de produtos da categoria na Silver
- `produtos_validos`, `produtos_invalidos`
- `pct_validos` — % de produtos válidos na categoria
- `pct_invalidos`
- Uma coluna `pct_falha_<regra>` para cada regra DQ (% de falha por categoria)
- `regra_mais_problematica` — nome da flag com maior % de falha na categoria
- `score_categoria` — média de score_registro dos produtos válidos
- `preco_medio_validos`, `preco_min_validos`, `preco_max_validos`
- `gold_generated_at`


In [0]:
flags_regra_cat = [c for c in df_silver.columns if c.startswith("r") and c.endswith("_falhou")]
n_regras_cat    = max(len(flags_regra_cat), 1)

# Agrega por categoria
agg_exprs = [
    F.count("*").cast("long").alias("total_produtos"),
    F.sum(F.when(F.col("silver_linha_valida") == True,  1).otherwise(0)).cast("long").alias("produtos_validos"),
    F.sum(F.when(F.col("silver_linha_valida") == False, 1).otherwise(0)).cast("long").alias("produtos_invalidos"),
]

# % de falha por regra
for flag in flags_regra_cat:
    agg_exprs.append(
        F.round(
            F.avg(F.when(F.col(flag), 1.0).otherwise(0.0)) * 100,
            4
        ).alias(f"pct_falha_{flag.replace('_falhou', '')}")
    )

# Métricas de preço (somente produtos válidos)
if "preco_lista" in df_silver.columns:
    agg_exprs += [
        F.round(F.avg(F.when(F.col("silver_linha_valida") == True, F.col("preco_lista"))), 2)
          .alias("preco_medio_validos"),
        F.min(F.when(F.col("silver_linha_valida") == True, F.col("preco_lista")))
          .alias("preco_min_validos"),
        F.max(F.when(F.col("silver_linha_valida") == True, F.col("preco_lista")))
          .alias("preco_max_validos"),
    ]

col_id_cat = "id_categoria" if "id_categoria" in df_silver.columns else "id_categoria_str"

df_agg_cat = (
    df_silver
    .groupBy(col_id_cat)
    .agg(*agg_exprs)
    .withColumn(
        "pct_validos",
        F.round(
            F.when(F.col("total_produtos") > 0,
                   (F.col("produtos_validos") / F.col("total_produtos")) * 100
            ).otherwise(F.lit(0.0)),
            4
        )
    )
    .withColumn(
        "pct_invalidos",
        F.round(F.lit(100.0) - F.col("pct_validos"), 4)
    )
)

# Identifica a regra mais problemática por categoria (maior pct_falha_*)
colunas_pct_falha = [f"pct_falha_{c.replace('_falhou','')}" for c in flags_regra_cat]
if colunas_pct_falha:
    # Usa array_max para encontrar a coluna com maior valor e extrair o nome
    df_gold_categoria = (
        df_agg_cat
        .withColumn(
            "_pct_array",
            F.array(*[
                F.struct(
                    F.col(c).alias("v"),
                    F.lit(c.replace("pct_falha_", "")).alias("n")
                ) for c in colunas_pct_falha
            ])
        )
        .withColumn("_max_struct", F.array_max(F.col("_pct_array")))
        .withColumn("regra_mais_problematica", F.col("_max_struct.n"))
        .drop("_pct_array", "_max_struct")
    )
else:
    df_gold_categoria = df_agg_cat.withColumn("regra_mais_problematica", F.lit(None).cast("string"))

df_gold_categoria = (
    df_gold_categoria
    .withColumn("gold_generated_at", F.current_timestamp())
    .orderBy(F.col("pct_invalidos").desc())
)

print(f"Categorias no perfil de DQ: {df_gold_categoria.count()}")
display(df_gold_categoria.limit(20))


## 11. Publicação: ADLS (Delta físico) + Azure SQL Server

In [0]:
print("=" * 70)
print("INICIANDO PUBLICAÇÃO DA CAMADA GOLD — ecommerce_produtos")
print("=" * 70)

# 1. DQ resumo por regra por dia
publicar_gold(df_gold_regra,      TABELA_GOLD_REGRA)

# 2. DQ resumo por tabela por hora
publicar_gold(df_gold_tabela,     TABELA_GOLD_TABELA)

# 3. Tendência diária de score de qualidade
publicar_gold(df_gold_tendencia,  TABELA_GOLD_TENDENCIA)

# 4. Regras críticas (acima do threshold)
publicar_gold(df_gold_criticas,   TABELA_GOLD_CRITICAS)

# 5. Snapshot analítico dos produtos válidos
publicar_gold(df_gold_snapshot,   TABELA_GOLD_SNAPSHOT)

# 6. Perfil de DQ por categoria
publicar_gold(df_gold_categoria,  TABELA_GOLD_CATEGORIA)

print("\n" + "=" * 70)
print(f"CAMADA GOLD FINALIZADA — RUN_ID: {RUN_ID}")
conta = STORAGE_ACCOUNT_NAME
print(
    f"Tabelas Delta em: "
    f"abfss://{CONTAINER_SQUAD1}@{conta}.dfs.core.windows.net/"
    f"{CAMADA_GOLD}/{ENTIDADE_GOLD}/"
)
print("Tabelas SQL Server em schema: squad1.*")
print("=" * 70)


## 12. Validação final — contagem de linhas e verificação física

In [0]:
tabelas_gold = {
    TABELA_GOLD_REGRA:     df_gold_regra,
    TABELA_GOLD_TABELA:    df_gold_tabela,
    TABELA_GOLD_TENDENCIA: df_gold_tendencia,
    TABELA_GOLD_CRITICAS:  df_gold_criticas,
    TABELA_GOLD_SNAPSHOT:  df_gold_snapshot,
    TABELA_GOLD_CATEGORIA: df_gold_categoria,
}

resumo = [(nome, df.count()) for nome, df in tabelas_gold.items()]
df_resumo = spark.createDataFrame(resumo, ["tabela_gold", "qtd_linhas"])
display(df_resumo)

# Verificação física dos paths Delta no ADLS
print("\nVerificação física dos caminhos Delta Gold:")
for nome in tabelas_gold:
    pasta = _pasta_fisica_gold(nome)
    existe = delta_existe_path(pasta)
    status = "✅" if existe else "❌"
    print(f"  {status} {pasta}")

print("\n✅ Validação concluída.")


### 13. Graficos de resultados - Análise em Data Quality

In [0]:
# ─── Gráfico 1: Resumo de falha por regra (Produtos) ──────────────────────
# Leitura da tabela: gold_ecommerce_produtos_dq_resumo_por_regra
# Gráfico: barras horizontais com % de falha por regra, ordenado decrescente.

import matplotlib.pyplot as plt
import pandas as pd

# 1. Parâmetros da tabela
pasta_gold_regra = _pasta_fisica_gold(TABELA_GOLD_REGRA)

# 2. Verifica se a tabela existe
if delta_existe_path(pasta_gold_regra):
    # 3. Lê os dados
    df_gold_regra = _ler_parquets_da_pasta(pasta_gold_regra)
    
    # 4. Converte para Pandas
    pdf_regra = df_gold_regra.toPandas()
    
    if not pdf_regra.empty:
        # 5. Ordena por pct_falha decrescente
        pdf_regra_sorted = pdf_regra.sort_values("pct_falha", ascending=False)
        
        # 6. Define cores por severidade
        cores = {"Critica": "#d9534f", "Aviso": "#f0ad4e"}
        pdf_regra_sorted["cor"] = pdf_regra_sorted["severidade"].map(cores)
        
        # 7. Cria o gráfico de barras horizontais
        fig, ax = plt.subplots(figsize=(10, 6))
        bars = ax.barh(
            pdf_regra_sorted["regra"],
            pdf_regra_sorted["pct_falha"],
            color=pdf_regra_sorted["cor"]
        )
        
        # 8. Adiciona rótulos e título
        ax.set_xlabel("% de Falha", fontsize=12)
        ax.set_title("Resumo de Falha por Regra de DQ - Produtos", fontsize=14, fontweight="bold")
        ax.invert_yaxis()  # Maior % no topo
        
        # 9. Adiciona valores ao lado das barras
        for bar in bars:
            width = bar.get_width()
            ax.text(
                width + 1,
                bar.get_y() + bar.get_height()/2,
                f"{width:.1f}%",
                va="center",
                fontsize=10
            )
        
        # 10. Ajusta layout e exibe
        plt.tight_layout()
        display(fig)
    else:
        print("⚠️ A tabela gold_ecommerce_produtos_dq_resumo_por_regra está vazia.")
else:
    print("⚠️ Tabela gold_ecommerce_produtos_dq_resumo_por_regra não encontrada.")

In [0]:
# ─── Gráfico 2: Resumo de registros limpos por tabela e hora (Produtos) ──────
# Leitura da tabela: gold_ecommerce_produtos_dq_resumo_por_tabela
# Gráfico: linhas com a evolução do percentual de registros limpos ao longo do tempo.

import matplotlib.pyplot as plt
import pandas as pd

# 1. Parâmetros da tabela
pasta_gold_tabela = _pasta_fisica_gold(TABELA_GOLD_TABELA)

# 2. Verifica se a tabela existe
if delta_existe_path(pasta_gold_tabela):
    # 3. Lê os dados
    df_gold_tabela = _ler_parquets_da_pasta(pasta_gold_tabela)
    
    # 4. Converte para Pandas
    pdf_tabela = df_gold_tabela.toPandas()
    
    if not pdf_tabela.empty:
        # 5. Ordena por data_hora
        pdf_tabela_sorted = pdf_tabela.sort_values("data_hora")
        
        # 6. Cria o gráfico de linhas
        fig, ax = plt.subplots(figsize=(12, 6))
        
        # Plota a porcentagem de limpos
        ax.plot(
            pdf_tabela_sorted["data_hora"],
            pdf_tabela_sorted["pct_limpos"],
            marker="o",
            linestyle="-",
            linewidth=2,
            markersize=8,
            color="#2c3e50",
            label="% Registros Limpos"
        )
        
        # 7. Adiciona título e rótulos
        ax.set_xlabel("Data/Hora", fontsize=12)
        ax.set_ylabel("% de Registros Limpos", fontsize=12)
        ax.set_title("Evolução da Qualidade da Tabela (Registros Limpos) - Produtos", fontsize=14, fontweight="bold")
        ax.grid(True, linestyle="--", alpha=0.6)
        ax.legend(loc="best")
        
        # 8. Adiciona rótulos de valor nos pontos
        for i, row in pdf_tabela_sorted.iterrows():
            ax.annotate(
                f"{row['pct_limpos']:.1f}%",
                (row["data_hora"], row["pct_limpos"]),
                textcoords="offset points",
                xytext=(0, 10),
                ha="center",
                fontsize=9
            )
        
        # 9. Ajusta layout e exibe
        plt.xticks(rotation=45)
        plt.tight_layout()
        display(fig)
    else:
        print("⚠️ A tabela gold_ecommerce_produtos_dq_resumo_por_tabela está vazia.")
else:
    print("⚠️ Tabela gold_ecommerce_produtos_dq_resumo_por_tabela não encontrada.")

In [0]:
# ─── Gráfico 3: Tendência diária do score de qualidade (Produtos) ─────────────
# Leitura da tabela: gold_ecommerce_produtos_dq_tendencia_diaria
# Gráfico: linha do score de qualidade com barras do delta (variação).

import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

# 1. Parâmetros da tabela
pasta_gold_tendencia = _pasta_fisica_gold(TABELA_GOLD_TENDENCIA)

# 2. Verifica se a tabela existe
if delta_existe_path(pasta_gold_tendencia):
    # 3. Lê os dados
    df_gold_tendencia = _ler_parquets_da_pasta(pasta_gold_tendencia)
    
    # 4. Converte para Pandas
    pdf_tendencia = df_gold_tendencia.toPandas()
    
    if not pdf_tendencia.empty:
        # 5. Ordena por data_execucao
        pdf_tendencia_sorted = pdf_tendencia.sort_values("data_execucao")
        
        # 6. Cria o gráfico com dois eixos
        fig, ax1 = plt.subplots(figsize=(12, 6))
        
        # Eixo primário: score de qualidade (linha)
        ax1.plot(
            pdf_tendencia_sorted["data_execucao"],
            pdf_tendencia_sorted["score_qualidade"],
            marker="o",
            linestyle="-",
            linewidth=2,
            markersize=8,
            color="#2c3e50",
            label="Score de Qualidade"
        )
        ax1.set_xlabel("Data", fontsize=12)
        ax1.set_ylabel("Score de Qualidade (0-100)", fontsize=12, color="#2c3e50")
        ax1.tick_params(axis="y", labelcolor="#2c3e50")
        ax1.grid(True, linestyle="--", alpha=0.6)
        
        # Adiciona rótulos do score
        for i, row in pdf_tendencia_sorted.iterrows():
            ax1.annotate(
                f"{row['score_qualidade']:.1f}",
                (row["data_execucao"], row["score_qualidade"]),
                textcoords="offset points",
                xytext=(0, 10),
                ha="center",
                fontsize=9,
                fontweight="bold"
            )
        
        # Eixo secundário: delta_score (barras)
        ax2 = ax1.twinx()
        # Define cores para delta: verde para positivo, vermelho para negativo
        cores_delta = [
            "#2ecc71" if d > 0 else "#e74c3c" if d < 0 else "#95a5a6"
            for d in pdf_tendencia_sorted["delta_score"].fillna(0)
        ]
        ax2.bar(
            pdf_tendencia_sorted["data_execucao"],
            pdf_tendencia_sorted["delta_score"].fillna(0),
            width=0.8,
            color=cores_delta,
            alpha=0.6,
            label="Variação (Δ)"
        )
        ax2.set_ylabel("Variação (Δ)", fontsize=12, color="#7f8c8d")
        ax2.tick_params(axis="y", labelcolor="#7f8c8d")
        
        # Adiciona rótulos do delta
        for i, row in pdf_tendencia_sorted.iterrows():
            if pd.notna(row["delta_score"]):
                ax2.annotate(
                    f"{row['delta_score']:+.1f}",
                    (row["data_execucao"], row["delta_score"]),
                    textcoords="offset points",
                    xytext=(0, -15 if row["delta_score"] >= 0 else 15),
                    ha="center",
                    fontsize=8,
                    color="#7f8c8d"
                )
        
        # 7. Título e legendas combinadas
        plt.title("Tendência Diária do Score de Qualidade - Produtos", fontsize=14, fontweight="bold")
        
        # Combina legendas
        lines1, labels1 = ax1.get_legend_handles_labels()
        lines2, labels2 = ax2.get_legend_handles_labels()
        ax1.legend(lines1 + lines2, labels1 + labels2, loc="best")
        
        plt.xticks(rotation=45)
        plt.tight_layout()
        display(fig)
    else:
        print("⚠️ A tabela gold_ecommerce_produtos_dq_tendencia_diaria está vazia.")
else:
    print("⚠️ Tabela gold_ecommerce_produtos_dq_tendencia_diaria não encontrada.")

In [0]:
# ─── Gráfico 4: Regras Críticas (Produtos) ────────────────────────────────────
# Leitura da tabela: gold_ecommerce_produtos_dq_regras_criticas
# Gráfico: barras horizontais com % de falha histórica, coloridas por status_alerta.

import matplotlib.pyplot as plt
import pandas as pd

# 1. Parâmetros da tabela
pasta_gold_criticas = _pasta_fisica_gold(TABELA_GOLD_CRITICAS)

# 2. Verifica se a tabela existe
if delta_existe_path(pasta_gold_criticas):
    # 3. Lê os dados
    df_gold_criticas = _ler_parquets_da_pasta(pasta_gold_criticas)
    
    # 4. Converte para Pandas
    pdf_criticas = df_gold_criticas.toPandas()
    
    if not pdf_criticas.empty:
        # 5. Ordena por pct_falha_historico decrescente
        pdf_criticas_sorted = pdf_criticas.sort_values("pct_falha_historico", ascending=False)
        
        # 6. Define cores por status_alerta
        cores_status = {
            "CRITICO": "#d9534f",   # vermelho
            "ATENCAO": "#f0ad4e",   # laranja/amarelo
            "OK": "#5cb85c"         # verde
        }
        pdf_criticas_sorted["cor"] = pdf_criticas_sorted["status_alerta"].map(cores_status)
        
        # 7. Cria o gráfico de barras horizontais
        fig, ax = plt.subplots(figsize=(12, 7))
        bars = ax.barh(
            pdf_criticas_sorted["regra"],
            pdf_criticas_sorted["pct_falha_historico"],
            color=pdf_criticas_sorted["cor"],
            height=0.6
        )
        
        # 8. Adiciona rótulos e título
        ax.set_xlabel("% de Falha Histórica", fontsize=12)
        ax.set_title("Regras Críticas de DQ - Produtos", fontsize=14, fontweight="bold")
        ax.invert_yaxis()  # Maior % no topo
        
        # 9. Ajusta o limite do eixo X
        max_val = pdf_criticas_sorted["pct_falha_historico"].max()
        ax.set_xlim(0, max_val * 1.15 if max_val > 0 else 10)
        
        # 10. Adiciona valores ao lado das barras com formatação legível
        for bar, row in zip(bars, pdf_criticas_sorted.itertuples()):
            width = bar.get_width()
            # Texto com % e status, sem negrito e com fundo branco para contraste
            label = f"{width:.1f}%"
            ax.text(
                width + (max_val * 0.02),
                bar.get_y() + bar.get_height()/2,
                label,
                va="center",
                fontsize=10,
                color="black",
                backgroundcolor="white",
                bbox=dict(facecolor="white", edgecolor="none", alpha=0.8, pad=1)
            )
        
        # 11. Adiciona anotação sobre o threshold
        ax.axvline(x=THRESHOLD_FALHA_CRITICA, color='red', linestyle='--', alpha=0.7, label=f'Threshold: {THRESHOLD_FALHA_CRITICA}%')
        ax.legend(loc='lower right')
        
        # 12. Ajusta layout e exibe
        plt.tight_layout()
        display(fig)
    else:
        print("⚠️ A tabela gold_ecommerce_produtos_dq_regras_criticas está vazia.")
else:
    print("⚠️ Tabela gold_ecommerce_produtos_dq_regras_criticas não encontrada.")

In [0]:
# ─── Gráfico 5: Snapshot de produtos válidos por faixa de preço (Produtos) ──
# Leitura da tabela: gold_ecommerce_produtos_snapshot_validos
# Gráfico: barras com a quantidade de produtos válidos por faixa de preço.

import matplotlib.pyplot as plt
import pandas as pd

# 1. Parâmetros da tabela
pasta_gold_snapshot = _pasta_fisica_gold(TABELA_GOLD_SNAPSHOT)

# 2. Verifica se a tabela existe
if delta_existe_path(pasta_gold_snapshot):
    # 3. Lê os dados
    df_gold_snapshot = _ler_parquets_da_pasta(pasta_gold_snapshot)
    
    # 4. Converte para Pandas
    pdf_snapshot = df_gold_snapshot.toPandas()
    
    if not pdf_snapshot.empty:
        # 5. Agrupa por faixa de preço
        agg_snapshot = pdf_snapshot.groupby("faixa_preco").agg(
            qtd_produtos=("sku", "count"),
            preco_medio=("preco_lista", "mean")
        ).reset_index()
        
        # Ordena as faixas na ordem correta
        ordem_faixas = ["Baixo", "Medio", "Alto", "Premium"]
        agg_snapshot["faixa_preco"] = pd.Categorical(agg_snapshot["faixa_preco"], categories=ordem_faixas, ordered=True)
        agg_snapshot = agg_snapshot.sort_values("faixa_preco")
        
        # 6. Cria o gráfico de barras
        fig, ax1 = plt.subplots(figsize=(10, 6))
        
        # Barras para quantidade de produtos
        bars = ax1.bar(
            agg_snapshot["faixa_preco"],
            agg_snapshot["qtd_produtos"],
            color=["#3498db", "#2ecc71", "#f39c12", "#e74c3c"],
            label="Quantidade de Produtos"
        )
        ax1.set_xlabel("Faixa de Preço", fontsize=12)
        ax1.set_ylabel("Quantidade de Produtos", fontsize=12, color="#2c3e50")
        ax1.tick_params(axis="y", labelcolor="#2c3e50")
        
        # Anota valores das barras
        for bar in bars:
            height = bar.get_height()
            ax1.annotate(
                f"{int(height)}",
                xy=(bar.get_x() + bar.get_width()/2, height),
                xytext=(0, 5),
                textcoords="offset points",
                ha="center",
                fontsize=10,
                color="black",
                backgroundcolor="white",
                bbox=dict(facecolor="white", edgecolor="none", alpha=0.8, pad=1)
            )
        
        # Eixo secundário para preço médio (linha)
        ax2 = ax1.twinx()
        ax2.plot(
            range(len(agg_snapshot)),
            agg_snapshot["preco_medio"],
            marker="o",
            linestyle="-",
            linewidth=2,
            markersize=8,
            color="#9b59b6",
            label="Preço Médio"
        )
        ax2.set_ylabel("Preço Médio (R$)", fontsize=12, color="#9b59b6")
        ax2.tick_params(axis="y", labelcolor="#9b59b6")
        
        # Adiciona valores do preço médio
        for i, row in agg_snapshot.iterrows():
            ax2.annotate(
                f"R${row['preco_medio']:.2f}",
                (i, row["preco_medio"]),
                textcoords="offset points",
                xytext=(0, 10),
                ha="center",
                fontsize=9,
                color="#9b59b6",
                backgroundcolor="white",
                bbox=dict(facecolor="white", edgecolor="none", alpha=0.8, pad=1)
            )
        
        # 7. Título e legenda
        plt.title("Produtos Válidos por Faixa de Preço", fontsize=14, fontweight="bold")
        
        # Combina legendas
        lines1, labels1 = ax1.get_legend_handles_labels()
        lines2, labels2 = ax2.get_legend_handles_labels()
        ax1.legend(lines1 + lines2, labels1 + labels2, loc="upper left")
        
        plt.tight_layout()
        display(fig)
    else:
        print("ℹ️ A tabela gold_ecommerce_produtos_snapshot_validos existe, mas está vazia (nenhum produto válido encontrado).")
else:
    print("⚠️ Tabela gold_ecommerce_produtos_snapshot_validos não encontrada.")

In [0]:
# ─── Gráfico 6: Perfil de DQ por categoria (Produtos) ────────────────────────
# Leitura da tabela: gold_ecommerce_produtos_dq_perfil_por_categoria
# Gráfico: barras horizontais com o percentual de produtos inválidos por categoria,
# destacando a regra mais problemática de cada uma.

import matplotlib.pyplot as plt
import pandas as pd

# 1. Parâmetros da tabela
pasta_gold_categoria = _pasta_fisica_gold(TABELA_GOLD_CATEGORIA)

# 2. Verifica se a tabela existe
if delta_existe_path(pasta_gold_categoria):
    # 3. Lê os dados
    df_gold_categoria = _ler_parquets_da_pasta(pasta_gold_categoria)
    
    # 4. Converte para Pandas
    pdf_categoria = df_gold_categoria.toPandas()
    
    if not pdf_categoria.empty:
        # 5. Filtra categorias com pelo menos 1 produto e ordena por percentual de inválidos
        pdf_categoria_filtrado = pdf_categoria[pdf_categoria["total_produtos"] > 0]
        pdf_categoria_sorted = pdf_categoria_filtrado.sort_values("pct_invalidos", ascending=False)
        
        # Limita a 20 categorias para não poluir o gráfico
        top_n = min(20, len(pdf_categoria_sorted))
        pdf_categoria_top = pdf_categoria_sorted.head(top_n)
        
        # 6. Cria o gráfico de barras horizontais
        fig, ax = plt.subplots(figsize=(12, 8))
        
        # Barras com percentual de inválidos
        bars = ax.barh(
            pdf_categoria_top["id_categoria"].astype(str),
            pdf_categoria_top["pct_invalidos"],
            color="#e74c3c",
            alpha=0.7,
            height=0.6
        )
        
        # 7. Adiciona rótulos e título
        ax.set_xlabel("% de Produtos Inválidos", fontsize=12)
        ax.set_ylabel("ID da Categoria", fontsize=12)
        ax.set_title("Top 20 Categorias com Maior Percentual de Produtos Inválidos", fontsize=14, fontweight="bold")
        ax.invert_yaxis()  # Maior % no topo
        
        # 8. Ajusta o limite do eixo X
        max_val = pdf_categoria_top["pct_invalidos"].max()
        ax.set_xlim(0, max_val * 1.2 if max_val > 0 else 10)
        
        # 9. Adiciona informações nas barras: % inválidos e regra mais problemática
        for i, (bar, row) in enumerate(zip(bars, pdf_categoria_top.itertuples())):
            width = bar.get_width()
            # Texto principal: % inválidos
            label1 = f"{width:.1f}% inválidos"
            ax.text(
                width + (max_val * 0.02),
                bar.get_y() + bar.get_height()/2 - 0.1,
                label1,
                va="center",
                fontsize=9,
                color="black",
                backgroundcolor="white",
                bbox=dict(facecolor="white", edgecolor="none", alpha=0.8, pad=1)
            )
            # Texto secundário: regra mais problemática
            label2 = f"Regra: {row.regra_mais_problematica if pd.notna(row.regra_mais_problematica) else 'N/A'}"
            ax.text(
                width + (max_val * 0.02),
                bar.get_y() + bar.get_height()/2 + 0.1,
                label2,
                va="center",
                fontsize=8,
                color="#7f8c8d"
            )
        
        # 10. Adiciona uma linha vertical no 50% como referência
        ax.axvline(x=50, color='red', linestyle='--', alpha=0.5, label='50% inválidos')
        ax.legend(loc='lower right')
        
        # 11. Ajusta layout e exibe
        plt.tight_layout()
        display(fig)
    else:
        print("ℹ️ A tabela gold_ecommerce_produtos_dq_perfil_por_categoria existe, mas está vazia.")
else:
    print("⚠️ Tabela gold_ecommerce_produtos_dq_perfil_por_categoria não encontrada.")